# Current legal dataset report

This report lists the law and legal-document titles currently represented in the processed article dataset. It also counts articles, identifies document types, and writes a CSV and JSON report for review.

In [ ]:
import csv
import json
from collections import defaultdict
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
REPORT_DIR = DATA_DIR / 'processed' / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

articles = [
    json.loads(line)
    for line in INPUT_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
print(f'Article records: {len(articles):,}')
print(f'Input: {INPUT_PATH}')

In [ ]:
documents = {}
for article in articles:
    document_id = str(article.get('document_id', '')).strip()
    title = str(article.get('document_title') or '').strip()
    item = documents.setdefault(document_id, {
        'document_id': document_id,
        'title': title,
        'document_type': article.get('document_type'),
        'source_url': article.get('source_url'),
        'article_count': 0,
        'article_ids': [],
    })
    item['article_count'] += 1
    item['article_ids'].append(article.get('id'))
    if not item['title'] and title:
        item['title'] = title

report = sorted(documents.values(), key=lambda item: (item['title'], item['document_id']))
for item in report:
    item['title_status'] = 'missing' if not item['title'] else 'present'
    item['duplicate_article_ids'] = len(item['article_ids']) != len(set(item['article_ids']))

print(f'Documents: {len(report):,}')
print(f'Documents with missing titles: {sum(item["title_status"] == "missing" for item in report):,}')
print(f'Total articles: {sum(item["article_count"] for item in report):,}')

In [ ]:
csv_path = REPORT_DIR / 'law_dataset_report.csv'
json_path = REPORT_DIR / 'law_dataset_report.json'
fieldnames = ['document_id', 'title', 'document_type', 'article_count', 'title_status', 'source_url']
with csv_path.open('w', encoding='utf-8-sig', newline='') as output:
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows({key: item.get(key) for key in fieldnames} for item in report)
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'CSV report: {csv_path}')
print(f'JSON report: {json_path}')

In [ ]:
print('Law and legal-document titles')
print('=' * 100)
for number, item in enumerate(report, start=1):
    print(f"{number:>4}. [{item['document_id']}] {item['title'] or '[MISSING TITLE]'} — {item['article_count']} articles")